# 05 · Solar-plus-battery mitigation for critical loads (3E)

**Question for this notebook:** how much PV and battery do Nagpur's hospitals and water-pumping stations need to ride through the outages found in notebooks 01–04, and how many critical service-hours does that buy per rupee?

| Step | Method | Main reference |
|---|---|---|
| Design outage | 95th percentile of the longest grid-off spell at each facility type (notebook 04) | — |
| Survival analysis | outage started at every hot-season hour in 20 years of NASA POWER data; a PV + battery size *survives* if it serves the essential load for the whole outage | Anderson et al. (2018) |
| Least-cost sizing | cheapest PV + battery combination with survival ≥ target | Laws et al. (2018); REopt (Cutler et al., 2017) |
| Benefit | rerun the cascade model with PV + battery at hospitals, pumps, or both → service-hours avoided | notebook 04 |
| Co-benefits | grid energy saved in normal operation, avoided CO₂ (Paris mitigation co-benefit) | CEA CO₂ baseline database |

**Run order:** last; needs notebook 02 (PV profile) and notebook 04 (facilities, outages, cascade parameters).

**Outputs** (in `results/05_mitigation/`): `survival_<archetype>.csv`, `sizing.csv`, `strategy_results.csv`, `cost_effectiveness.csv`, figures.

> **Cost and load figures are assumptions.** Check them against the latest MNRE benchmark cost order, current battery quotations, and facility energy audits before quoting results.

## 0. Setup

In [ ]:
!pip install -q pandapower

In [ ]:
import os, sys, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})

# ---- Shared project settings (identical in all five notebooks) ----
PROJECT_ID = "goggle-earth-499404"          # Google Cloud project registered for Earth Engine
SITE = "Nagpur, Maharashtra"
LAT, LON, ELEV_M = 21.15, 79.09, 310
TZ = "Asia/Kolkata"

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True                             # keep results in Google Drive so the notebooks can share them
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/grid_stress_nagpur"
else:
    ROOT = os.path.abspath("grid_stress_nagpur")

CACHE = os.path.join(ROOT, "cache")
RES = {k: os.path.join(ROOT, "results", k) for k in
       ["01_contingency", "02_feeder", "03_heat", "04_cascade", "05_mitigation"]}
for d in [CACHE, *RES.values()]:
    os.makedirs(d, exist_ok=True)
OUT = RES["05_mitigation"]

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, name), dpi=150, bbox_inches="tight")
    plt.show()

def require(path, notebook):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run notebook {notebook} first (with USE_DRIVE = True).")
    return path

print("Project folder:", ROOT)

## 1. Parameters
**Essential loads** are the loads that must stay on during an outage: ICU, operating theatres, emergency, oxygen concentrators, cold chain, lighting, minimum cooling, and the hospital's own pumps. The per-bed values below are placeholders consistent with the WHO/World Bank *Energizing Health* (2023) facility tiers and Indian hospital audits; replace them with audit data where possible.

**Water pumping** is sized for an emergency supply of `LPCD_ESSENTIAL` litres per person per day (well above the Sphere minimum of 15 l). Pumping energy depends on head; 0.4 kWh/m³ is a typical urban figure.

**Costs (INR, verify before quoting):**
- PV: the MNRE benchmark for 10–100 kWp rooftop systems is in the ₹40–50k/kWp range (check the current year's order).
- LFP battery, installed: ₹18–25k/kWh.
- Islanding hybrid inverter / PCS: per kW of peak essential load.

In [ ]:
SEASON = (3, 6)                       # outages simulated in March–June (storms and heat waves)
START_STEP_H = 3                      # every 3rd hour of the season is an outage start
TARGET_SURVIVAL = 0.95                # share of start times the system must survive
DESIGN_PCT = 95                       # design outage = this percentile of the longest grid-off spells
DEFAULT_DESIGN_H = 24                 # used if a facility type never loses grid power in the scenarios
SOC_START = 1.0                       # battery kept full for backup (resilience reserve)

PV_GRID = np.arange(0, 10.01, 0.5)    # kWp per kW of average essential load
BATT_GRID = np.arange(0, 96.01, 4)    # kWh per kW of average essential load (= hours of average load)

ESSENTIAL_KWH_PER_BED_DAY = {"major": 6.0, "small": 4.0}
LPCD_ESSENTIAL = 40                   # emergency water supply, litres per person per day
PUMP_KWH_PER_M3 = 0.4

# Hourly shapes of essential load (fraction of daily mean), IST hours 0..23
HOSP_SHAPE = np.array([0.80, 0.78, 0.76, 0.75, 0.75, 0.78, 0.88, 1.00, 1.12, 1.20, 1.25, 1.27,
                       1.27, 1.25, 1.22, 1.18, 1.12, 1.05, 1.00, 0.97, 0.93, 0.90, 0.86, 0.82])
HOSP_SHAPE = HOSP_SHAPE / HOSP_SHAPE.mean()
PUMP_SHAPE = np.ones(24)              # emergency pumping spread over the day

PV_CAPEX = 45000                      # INR per kWp  (MNRE benchmark range — verify)
BATT_CAPEX = 22000                    # INR per kWh  (LFP, installed — verify)
PCS_CAPEX = 15000                     # INR per kW of peak essential load (hybrid / islanding inverter)
LIFE = {"pv": 25, "batt": 10, "pcs": 10}
DISCOUNT = 0.09
OM_FRAC = 0.015                       # annual O&M as a share of capex
TARIFF = 8.0                          # INR/kWh avoided grid purchase in normal operation (commercial tariff, assumption)
EXPORT_TARIFF = 3.0                   # INR/kWh for PV energy exported beyond the facility's own use (assumption)
ESSENTIAL_SHARE = 0.4                 # essential load as a share of the facility's total consumption
GRID_EF = 0.71                        # tCO2/MWh, CEA CO2 baseline database (check latest version)

N_STARTS_BENEFIT = 8                  # random outage start times per scenario in the benefit analysis
SEED = 11
rng = np.random.default_rng(SEED)

## 2. Inputs from notebooks 02 and 04

In [ ]:
pv = pd.read_csv(require(os.path.join(RES["02_feeder"], "pv_profile_per_kwp.csv"), "02"), index_col="time_utc", parse_dates=True)
pv["ist"] = pv.index + pd.Timedelta(hours=5, minutes=30)
PV = pv["pv_kw_per_kwp"].values
HOUR = pv["ist_hour"].values

fac = pd.read_csv(require(os.path.join(RES["04_cascade"], "facilities.csv"), "04"))
fout = pd.read_csv(os.path.join(RES["04_cascade"], "facility_outages.csv"))
cp = json.load(open(os.path.join(RES["04_cascade"], "cascade_params.json")))
P, TAIL_H = cp["P"], cp["TAIL_H"]

# essential load per facility (average kW)
fac["archetype"] = np.where(fac.type == "pump", "pump", fac.tier + " hospital")
fac["mean_kw"] = np.where(fac.type == "pump",
                          fac["pop"] * LPCD_ESSENTIAL / 1000 * PUMP_KWH_PER_M3 / 24,
                          fac.beds * fac.tier.map(ESSENTIAL_KWH_PER_BED_DAY).fillna(4.0) / 24)
SHAPES = {"major hospital": HOSP_SHAPE, "small hospital": HOSP_SHAPE, "pump": PUMP_SHAPE}
fac.groupby("archetype").agg(facilities=("id", "size"), mean_kw_each=("mean_kw", "mean"), total_kw=("mean_kw", "sum"))

## 3. Design outage durations
For each archetype: the 95th percentile of the longest continuous grid-off spell across all scenarios in which that facility type lost grid supply. Storm restoration (notebook 01) drives the long spells.

In [ ]:
fout["archetype"] = fout.facility.map(fac.set_index("id")["archetype"])
DESIGN_H = {}
for a in SHAPES:
    d = fout[fout.archetype == a].longest_spell_h
    DESIGN_H[a] = int(np.ceil(np.percentile(d, DESIGN_PCT))) if len(d) else DEFAULT_DESIGN_H
D_MAX = int(min(max(max(DESIGN_H.values()), 72), 96))
print("Design outage (h):", DESIGN_H, "| simulated up to", D_MAX, "h")

fig, ax = plt.subplots(figsize=(7, 3.2))
for a, d in fout.groupby("archetype"):
    s = np.sort(d.longest_spell_h); ax.step(s, np.arange(1, len(s) + 1) / len(s), where="post", label=a)
ax.set_xlabel("Longest grid-off spell (h)"); ax.set_ylabel("Cumulative share of outages"); ax.legend(frameon=False)
ax.set_title("Outage durations seen by critical facilities (notebook 04)")
savefig("fig01_outage_durations.png")

## 4. Survival analysis
For a system with \(x\) kWp of PV and \(y\) kWh of battery per kW of average essential load, an outage starting at hour \(t_0\) is simulated hour by hour:

$$SOC_{t+1} = \min\!\left(y,\; SOC_t + x\,p_{t} - \ell_{t}\right), \qquad SOC_{t_0} = y \cdot SOC_{start}$$

where \(p_t\) is PV output per kWp (notebook 02) and \(\ell_t\) is essential load per kW of average load. The system **survives** an outage of \(D\) hours if \(SOC\) never drops below zero within the first \(D\) hours. Survival probability is the share of all March–June start times (every `START_STEP_H` hours, 2005–2024) that survive (Anderson et al., 2018). Because everything is normalised per kW of load, one simulation per load shape covers every facility of that archetype.

In [ ]:
season_mask = (pv.ist.dt.month >= SEASON[0]) & (pv.ist.dt.month <= SEASON[1])
STARTS = np.where(season_mask.values)[0]
STARTS = STARTS[STARTS + D_MAX < len(PV)][::START_STEP_H]
print(f"{len(STARTS):,} outage start times")

def survival_table(shape):
    load = shape[HOUR]
    fail = np.full((len(PV_GRID), len(BATT_GRID), len(STARTS)), D_MAX + 1, dtype=np.int16)
    cap = BATT_GRID[:, None].astype(float)
    for i, x in enumerate(PV_GRID):
        soc = np.repeat(cap * SOC_START, len(STARTS), axis=1)
        alive = np.ones_like(soc, bool)
        for t in range(D_MAX):
            idx = STARTS + t
            soc = np.minimum(cap, soc + (x * PV[idx] - load[idx])[None, :])
            now = alive & (soc < -1e-9)
            fail[i][now] = t
            alive &= ~now
    return fail

FAIL, SURV = {}, {}
t0 = time.time()
for a in ["major hospital", "pump"] if np.allclose(SHAPES["major hospital"], SHAPES["small hospital"]) else SHAPES:
    FAIL[a] = survival_table(SHAPES[a])
if "small hospital" not in FAIL:
    FAIL["small hospital"] = FAIL["major hospital"]          # identical normalised shape
def survival(a, D):
    return (FAIL[a] >= D).mean(axis=2)                       # (n_pv, n_batt)
print(f"Survival tables in {time.time() - t0:.0f}s")

## 5. Least-cost sizing
Annualised cost per kW of average essential load:

$$C(x, y) = x\,c_{PV}\,(CRF_{25} + o) + y\,c_{batt}\,(CRF_{10} + o) + \hat\ell\,c_{PCS}\,(CRF_{10} + o)$$

with capital recovery factor \(CRF_n = \frac{r(1+r)^n}{(1+r)^n - 1}\), O&M share \(o\), and peak-to-average load ratio \(\hat\ell\). We pick the cheapest \((x, y)\) whose survival at the design outage is at least `TARGET_SURVIVAL`.

In [ ]:
crf = lambda n: DISCOUNT * (1 + DISCOUNT) ** n / ((1 + DISCOUNT) ** n - 1)
XX, YY = np.meshgrid(PV_GRID, BATT_GRID, indexing="ij")
def annual_cost_per_kw(x, y, shape):
    return (x * PV_CAPEX * (crf(LIFE["pv"]) + OM_FRAC) + y * BATT_CAPEX * (crf(LIFE["batt"]) + OM_FRAC)
            + shape.max() * PCS_CAPEX * (crf(LIFE["pcs"]) + OM_FRAC))
def capex_per_kw(x, y, shape):
    return x * PV_CAPEX + y * BATT_CAPEX + shape.max() * PCS_CAPEX

yield_kwh_per_kwp = pv.groupby(pv.ist.dt.year)["pv_kw_per_kwp"].sum().iloc[1:-1].mean()
sizing = []
for a, shape in SHAPES.items():
    D = DESIGN_H[a]
    s = survival(a, D)
    cost = annual_cost_per_kw(XX, YY, shape)
    ok = s >= TARGET_SURVIVAL
    if ok.any():
        k = np.unravel_index(np.where(ok, cost, np.inf).argmin(), cost.shape)
    else:
        print(f"WARNING: no size reaches {TARGET_SURVIVAL:.0%} survival for {a} at {D} h; using the best available")
        k = np.unravel_index(s.argmax(), s.shape)
    x, y = PV_GRID[k[0]], BATT_GRID[k[1]]
    sel = fac[fac.archetype == a]
    sizing.append(dict(archetype=a, facilities=len(sel), design_outage_h=D,
                       pv_kwp_per_kw=x, batt_kwh_per_kw=y, survival=s[k],
                       mean_kw_each=sel.mean_kw.mean(), pv_kwp_each=x * sel.mean_kw.mean(), batt_kwh_each=y * sel.mean_kw.mean(),
                       capex_inr_total=capex_per_kw(x, y, shape) * sel.mean_kw.sum(),
                       annual_cost_inr_total=annual_cost_per_kw(x, y, shape) * sel.mean_kw.sum(),
                       annual_pv_kwh_total=x * sel.mean_kw.sum() * yield_kwh_per_kwp))
sizing = pd.DataFrame(sizing).set_index("archetype")
total_use_kwh = pd.Series({a: fac[fac.archetype == a].mean_kw.sum() * 8760 / ESSENTIAL_SHARE for a in SHAPES})
self_used = np.minimum(sizing.annual_pv_kwh_total, total_use_kwh)
sizing["annual_energy_saving_inr"] = self_used * TARIFF + (sizing.annual_pv_kwh_total - self_used) * EXPORT_TARIFF
sizing["annual_co2_avoided_t"] = sizing.annual_pv_kwh_total / 1000 * GRID_EF
sizing["net_annual_cost_inr"] = sizing.annual_cost_inr_total - sizing.annual_energy_saving_inr
sizing.to_csv(os.path.join(OUT, "sizing.csv"))
for a in SHAPES:
    pd.DataFrame(survival(a, DESIGN_H[a]), index=PV_GRID, columns=BATT_GRID).to_csv(
        os.path.join(OUT, f"survival_{a.replace(' ', '_')}.csv"))
sizing.T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, a in zip(axes, SHAPES):
    s = survival(a, DESIGN_H[a])
    im = ax.contourf(BATT_GRID, PV_GRID, s, levels=[0, .5, .8, .9, .95, .99, 1.0], cmap="Greens")
    ax.contour(BATT_GRID, PV_GRID, annual_cost_per_kw(XX, YY, SHAPES[a]) / 1000, colors="grey", linewidths=0.6, levels=8)
    r = sizing.loc[a]; ax.plot(r.batt_kwh_per_kw, r.pv_kwp_per_kw, "r*", ms=12)
    ax.set_xlabel("Battery (kWh per kW of load)"); ax.set_ylabel("PV (kWp per kW of load)")
    ax.set_title(f"{a}: survival of a {DESIGN_H[a]} h outage", fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.8, label="survival probability")
plt.savefig(os.path.join(OUT, "fig02_survival_maps.png"), dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Survival vs outage duration for the chosen designs
Ds = np.arange(1, D_MAX + 1)
fig, ax = plt.subplots(figsize=(7, 3.4))
for a in SHAPES:
    r = sizing.loc[a]
    i, j = list(PV_GRID).index(r.pv_kwp_per_kw), list(BATT_GRID).index(r.batt_kwh_per_kw)
    ax.plot(Ds, [(FAIL[a][i, j] >= D).mean() for D in Ds], label=f"{a} ({r.pv_kwp_per_kw:g} kWp, {r.batt_kwh_per_kw:g} kWh per kW)")
    ax.axvline(DESIGN_H[a], ls=":", lw=0.8, color=ax.lines[-1].get_color())
ax.axhline(TARGET_SURVIVAL, color="grey", lw=0.8)
ax.set_xlabel("Outage duration (h)"); ax.set_ylabel("Survival probability"); ax.legend(frameon=False, fontsize=7)
savefig("fig03_survival_curves.png")

## 6. Benefit: critical service-hours avoided
The cascade model of notebook 04 is rerun with the sized PV + battery systems as an extra power source at (a) hospitals only, (b) pumping stations only, (c) both. Order of supply at a hospital: grid → PV + battery → diesel. Each scenario starts at `N_STARTS_BENEFIT` random March–June times, so the PV output during the outage varies; results are averaged.

In [ ]:
# ---- Cascade model: power -> water -> hospitals (shared by notebooks 04 and 05) ----
import networkx as nx


def build_dependency_graph(fac):
    """Directed dependency graph. Edge u -> v means 'v needs u'.
    bus -> pump (power), bus -> hospital (power), pump -> hospital (water), pump -> water zone (water)."""
    G = nx.DiGraph()
    for b in sorted(fac.bus.unique()):
        G.add_node(f"bus{b}", kind="bus", bus=int(b), layer=0)
    for _, f in fac[fac.type == "pump"].iterrows():
        G.add_node(f["id"], kind="pump", layer=1)
        G.add_edge(f"bus{f['bus']}", f["id"], dep="power")
        G.add_node(f"zone_{f['id']}", kind="water_zone", pop=float(f["pop"]), layer=2)
        G.add_edge(f["id"], f"zone_{f['id']}", dep="water")
    for _, f in fac[fac.type == "hospital"].iterrows():
        G.add_node(f["id"], kind="hospital", tier=f["tier"], beds=float(f["beds"]), layer=2)
        G.add_edge(f"bus{f['bus']}", f["id"], dep="power")
        G.add_edge(f["pump"], f["id"], dep="water")
    assert nx.is_directed_acyclic_graph(G)
    return G


def dependency_arrays(G):
    """Turn the graph into index arrays, processed layer by layer (topological generations)."""
    nodes = {k: [n for n, d in G.nodes(data=True) if d["kind"] == k] for k in ["bus", "pump", "hospital", "water_zone"]}
    pos = {k: {n: i for i, n in enumerate(v)} for k, v in nodes.items()}
    def pred(n, kind):
        return next(u for u in G.predecessors(n) if G.nodes[u]["kind"] == kind)
    arr = dict(
        nodes=nodes,
        bus_ids=np.array([G.nodes[n]["bus"] for n in nodes["bus"]]),
        pump_bus=np.array([pos["bus"][pred(p, "bus")] for p in nodes["pump"]]),
        hosp_bus=np.array([pos["bus"][pred(h, "bus")] for h in nodes["hospital"]]),
        hosp_pump=np.array([pos["pump"][pred(h, "pump")] for h in nodes["hospital"]]),
        zone_pump=np.array([pos["pump"][pred(z, "pump")] for z in nodes["water_zone"]]),
        zone_pop=np.array([G.nodes[z]["pop"] for z in nodes["water_zone"]]),
        hosp_beds=np.array([G.nodes[h]["beds"] for h in nodes["hospital"]]),
        hosp_tier=np.array([G.nodes[h]["tier"] for h in nodes["hospital"]]),
        generations=[sorted({G.nodes[n]["kind"] for n in g}) for g in nx.topological_generations(G)],
    )
    return arr


def shed_fraction_matrix(ts_scenario, bus_ids, bus_load, horizon):
    """Hourly fraction of each bus's load that is shed (T x n_bus)."""
    F = np.zeros((horizon, len(bus_ids)))
    if len(ts_scenario):
        piv = ts_scenario.pivot_table(index="hour", columns="bus", values="shed_mw", aggfunc="sum")
        piv = piv.reindex(index=range(horizon), columns=bus_ids).fillna(0.0)
        F = np.clip(piv.values / np.maximum(bus_load.reindex(bus_ids).values, 1e-9), 0, 1)
    return F


def _battery_step(on_grid, soc, cap, pv_kw, load_kw, charge_rate):
    """One hour of PV + battery. Returns (supplied, new_soc). Grid-connected: battery recharges."""
    surplus = pv_kw - load_kw
    soc_off = soc + np.where(surplus >= 0, np.minimum(surplus, cap - soc), surplus)
    ok_off = soc_off >= -1e-9
    soc_off = np.clip(soc_off, 0, cap)
    soc_on = np.minimum(cap, soc + charge_rate * cap)
    return np.where(on_grid, True, ok_off), np.where(on_grid, soc_on, soc_off)


def cascade_simulate(A, F, P, backup=None):
    """Hourly simulation of facility states for one scenario.

    A: dependency arrays; F: shed fraction (T x n_bus); P: parameters;
    backup: optional {"hospital"|"pump": dict(pv_kwp, batt_kwh, load_kw (T x n), pv (T,))}.
    Returns per-facility hourly states."""
    T = F.shape[0]
    npump, nh = len(A["pump_bus"]), len(A["hosp_bus"])
    pump_grid = F[:, A["pump_bus"]] < P["PUMP_SHED_THRESHOLD"]
    hosp_grid = F[:, A["hosp_bus"]] < P["HOSP_SHED_THRESHOLD"]

    def run_backup(kind, grid):
        if not backup or kind not in backup:
            return np.zeros_like(grid)
        b = backup[kind]
        cap = b["batt_kwh"].astype(float)
        soc = cap * P.get("SOC_START", 1.0)
        sup = np.zeros_like(grid)
        for t in range(T):
            pvk = b["pv_kwp"] * b["pv"][t]
            s, soc = _battery_step(grid[t], soc, cap, pvk, b["load_kw"][t], P.get("CHARGE_RATE", 0.5))
            sup[t] = s & ~grid[t]
        return sup

    pump_backup = run_backup("pump", pump_grid)
    pfuel = np.full(npump, float(P.get("PUMP_DG_H", 0)))

    # water network storage (elevated service reservoirs), hours of demand
    S = np.full(npump, float(P["WATER_STORAGE_H"]))
    pump_on = np.zeros((T, npump), bool)
    zone_water = np.zeros((T, npump), bool)
    for t in range(T):
        g = pump_grid[t] | pump_backup[t]
        dg = ~g & (pfuel > 0)
        pfuel = np.where(dg, pfuel - 1, pfuel)
        on = g | dg
        pump_on[t] = on
        zone_water[t] = on | (S > 0)
        S = np.where(on, np.minimum(P["WATER_STORAGE_H"], S + P["REFILL_PER_H"]), np.maximum(S - 1, 0))

    hosp_backup = run_backup("hospital", hosp_grid)
    fuel = np.where(A["hosp_tier"] == "major", P["DG_HOURS_MAJOR"], P["DG_HOURS_SMALL"]).astype(float)
    W = np.full(nh, float(P["HOSP_WATER_H"]))
    hp, hw = np.zeros((T, nh), bool), np.zeros((T, nh), bool)
    for t in range(T):
        g = hosp_grid[t] | hosp_backup[t]
        dg = ~g & (fuel > 0)
        fuel = np.where(dg, fuel - 1, fuel)
        hp[t] = g | dg
        zw = zone_water[t, A["hosp_pump"]]
        hw[t] = zw | (W > 0)
        W = np.where(zw, np.minimum(P["HOSP_WATER_H"], W + P["REFILL_PER_H"]), np.maximum(W - 1, 0))
    return dict(pump_grid=pump_grid, pump_on=pump_on, zone_water=zone_water,
                hosp_grid=hosp_grid, hosp_power=hp, hosp_water=hw, hosp_ok=hp & hw)


def cascade_metrics(A, R):
    down = ~R["hosp_ok"]
    power_down = ~R["hosp_power"]
    water_only = down & ~power_down
    zone_down = ~R["zone_water"]
    return dict(
        hospitals_grid_off=int((~R["hosp_grid"]).any(0).sum()),
        hospitals_affected=int(down.any(0).sum()),
        hospital_hours_lost=float(down.sum()),
        hospital_hours_power=float((down & power_down).sum()),
        hospital_hours_water=float(water_only.sum()),
        bed_hours_lost=float((down * A["hosp_beds"]).sum()),
        pumps_grid_off=int((~R["pump_grid"]).any(0).sum()),
        water_zone_hours_lost=float(zone_down.sum()),
        person_hours_without_water=float((zone_down * A["zone_pop"][None, :]).sum()),
        critical_service_hours_lost=float(down.sum() + zone_down.sum()),
    )

In [ ]:
G = build_dependency_graph(fac)
A = dependency_arrays(G)
buses = pd.read_csv(os.path.join(RES["01_contingency"], "network_buses.csv"), index_col="bus")
meta = pd.read_csv(os.path.join(RES["01_contingency"], "scenarios_meta.csv"))
ts = pd.read_csv(os.path.join(RES["01_contingency"], "bus_shed_timeseries.csv"))
if os.path.exists(os.path.join(RES["03_heat"], "heat_scenarios_meta.csv")):
    meta = pd.concat([meta, pd.read_csv(os.path.join(RES["03_heat"], "heat_scenarios_meta.csv"))], ignore_index=True)
    ts = pd.concat([ts, pd.read_csv(os.path.join(RES["03_heat"], "heat_bus_shed_timeseries.csv"))], ignore_index=True)
TS = {k: v for k, v in ts.groupby("scenario_id")}
affected = set(fout.scenario_id)                      # scenarios where some facility lost grid power
meta = meta[meta.scenario_id.isin(affected)]

fi = fac.set_index("id")
H = fi.loc[A["nodes"]["hospital"]]; W = fi.loc[A["nodes"]["pump"]]
def design(rows):
    r = sizing.loc[rows.archetype]
    return rows.mean_kw.values * r.pv_kwp_per_kw.values, rows.mean_kw.values * r.batt_kwh_per_kw.values
H_pv, H_batt = design(H); W_pv, W_batt = design(W)

def backup_for(strategy, start, horizon):
    sl = slice(start, start + horizon)
    hrs, pvs = HOUR[sl], PV[sl]
    b = {}
    if strategy in ("hospitals", "both"):
        b["hospital"] = dict(pv_kwp=H_pv, batt_kwh=H_batt, pv=pvs, load_kw=HOSP_SHAPE[hrs][:, None] * H.mean_kw.values[None, :])
    if strategy in ("pumps", "both"):
        b["pump"] = dict(pv_kwp=W_pv, batt_kwh=W_batt, pv=pvs, load_kw=PUMP_SHAPE[hrs][:, None] * W.mean_kw.values[None, :])
    return b

P5 = dict(P, SOC_START=SOC_START)
season_starts = np.where(season_mask.values)[0]
season_starts = season_starts[season_starts + 200 < len(PV)]
rows = []
t0 = time.time()
for _, m in meta.iterrows():
    t = TS.get(m.scenario_id, pd.DataFrame(columns=["hour", "bus", "shed_mw"]))
    horizon = int(t.hour.max() + 1 + TAIL_H) if len(t) else TAIL_H
    F = shed_fraction_matrix(t, A["bus_ids"], buses.load_mw, horizon)
    base = cascade_metrics(A, cascade_simulate(A, F, P5))
    rows.append(dict(scenario_id=m.scenario_id, family=m.family, strategy="none", **base))
    for strategy in ["hospitals", "pumps", "both"]:
        acc = []
        for st in rng.choice(season_starts, N_STARTS_BENEFIT, replace=False):
            acc.append(cascade_metrics(A, cascade_simulate(A, F, P5, backup_for(strategy, st, horizon))))
        rows.append(dict(scenario_id=m.scenario_id, family=m.family, strategy=strategy,
                         **pd.DataFrame(acc).mean().to_dict()))
res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT, "strategy_results.csv"), index=False)
print(f"{len(meta)} scenarios x 4 strategies in {time.time() - t0:.0f}s")

## 7. Cost-effectiveness

In [ ]:
cost = {"none": 0.0,
        "hospitals": sizing.loc[["major hospital", "small hospital"], "annual_cost_inr_total"].sum(),
        "pumps": sizing.loc["pump", "annual_cost_inr_total"],
        }
cost["both"] = cost["hospitals"] + cost["pumps"]
saving = {"none": 0.0,
          "hospitals": sizing.loc[["major hospital", "small hospital"], "annual_energy_saving_inr"].sum(),
          "pumps": sizing.loc["pump", "annual_energy_saving_inr"]}
saving["both"] = saving["hospitals"] + saving["pumps"]
co2 = {"none": 0.0, "hospitals": sizing.loc[["major hospital", "small hospital"], "annual_co2_avoided_t"].sum(),
       "pumps": sizing.loc["pump", "annual_co2_avoided_t"]}
co2["both"] = co2["hospitals"] + co2["pumps"]

piv = res.pivot_table(index=["family", "scenario_id"], columns="strategy", values="critical_service_hours_lost")
avoided = piv[["hospitals", "pumps", "both"]].rsub(piv["none"], axis=0)
by_family = avoided.groupby("family").mean()
print("Mean critical service-hours avoided per event:")
display(by_family.round(1))

ce = pd.DataFrame({
    "annualised cost (₹ lakh/yr)": pd.Series(cost) / 1e5,
    "energy saving (₹ lakh/yr)": pd.Series(saving) / 1e5,
    "net cost (₹ lakh/yr)": (pd.Series(cost) - pd.Series(saving)) / 1e5,
    "CO2 avoided (t/yr)": pd.Series(co2),
    "service-hours avoided, mean storm event": avoided[avoided.index.get_level_values(0).str.startswith("storm")].mean(),
    "service-hours avoided, all events": avoided.mean(),
    "hospital-hours avoided, mean storm event": res[res.family.str.startswith("storm")].pivot_table(index="scenario_id", columns="strategy", values="hospital_hours_lost").pipe(lambda d: d[["hospitals", "pumps", "both"]].rsub(d["none"], axis=0)).mean(),
}).drop(index="none")
ce["annual cost per service-hour avoided in one storm event (₹)"] = ce["annualised cost (₹ lakh/yr)"] * 1e5 / ce["service-hours avoided, mean storm event"].replace(0, np.nan)
ce.to_csv(os.path.join(OUT, "cost_effectiveness.csv"))
ce.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
c = {"hospitals": "#c0392b", "pumps": "#2c6fbb", "both": "#6a3d9a"}
fam_order = by_family.index.tolist()
xx = np.arange(len(fam_order)); wdt = 0.27
for k, s in enumerate(["hospitals", "pumps", "both"]):
    ax[0].bar(xx + (k - 1) * wdt, by_family[s], wdt, color=c[s], label=f"PV+battery at {s}")
ax[0].set_xticks(xx); ax[0].set_xticklabels(fam_order, rotation=30, ha="right", fontsize=7)
ax[0].set_ylabel("Critical service-hours avoided per event"); ax[0].legend(frameon=False, fontsize=7)
for s in ["hospitals", "pumps", "both"]:
    ax[1].scatter(ce.loc[s, "annualised cost (₹ lakh/yr)"], ce.loc[s, "service-hours avoided, mean storm event"], s=80, color=c[s], label=s)
    ax[1].scatter(ce.loc[s, "net cost (₹ lakh/yr)"], ce.loc[s, "service-hours avoided, mean storm event"], s=80, facecolors="none", edgecolors=c[s])
ax[1].axvline(0, color="grey", lw=0.8)
ax[1].set_xlabel("Annual cost, ₹ lakh/yr (filled: gross; hollow: net of energy savings)")
ax[1].set_ylabel("Service-hours avoided per storm event"); ax[1].legend(frameon=False, fontsize=7)
savefig("fig04_cost_effectiveness.png")

**Reading the results.**
- *Hospitals only* removes direct power loss, but hospitals can still fail on water when pumps stop.
- *Pumps only* protects both water zones and the hospitals' water supply, so it acts on the cascade.
- *Both* captures the interaction between the two.
- If the net cost (after energy savings) is negative, the systems pay for themselves in normal operation and the resilience benefit is a free co-benefit: a "no-regret", Build-Back-Better option (Sendai Priority 4) with Paris mitigation co-benefits (avoided CO₂).

## 8. Notes for the technical note
- **Method:** survival-based sizing over 20 years of hourly NASA POWER data (Anderson et al., 2018). The design outage comes from grid contingency and storm restoration analysis, not an assumed 24/72 h. Benefits are measured with the cascade model.
- **Limitations:**
  - assumed essential loads and costs
  - one load shape per archetype
  - perfect battery availability and no degradation
  - roof area not checked (large PV sizes may not fit)
  - events are not given annual probabilities, so cost-effectiveness is per event; add event frequencies for an expected annual benefit
  - assumed DG behaviour
- **Sensitivity worth reporting:** `TARGET_SURVIVAL` (0.9/0.99), `BATT_CAPEX` ±30%, `ESSENTIAL_KWH_PER_BED_DAY`, `LPCD_ESSENTIAL`.